# Meta Model — Directional Prediction (Q25 / Q50 / Q75)

**Goal:** predict the sign and magnitude of the 8h log return, conditioned on bars where the MFE Q50 model already signals a large move (Q50_mfe ≥ 30).

**Pipeline:**
1. Load all `features_9` parquets + 1H OHLCV (for the 8h log return target)
2. Filter training rows to `Q50_mfe ≥ 30` (MFE model forward pass on train set — no leakage because MFE model is fixed)
3. Same feature set as MFE model (308 features, same VOL_DROP exclusions)
4. Target: `ret_8h = log(close_{t+8} / close_t)`
5. Walk-forward expanding CV (same 5-fold structure as MFE training)
6. Train Q25, Q50, Q75 LightGBM quantile regressors
7. Evaluate: OOF coverage, sign accuracy, correlation, per-pair breakdown

In [6]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit

import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
FEATURES_DIR  = Path('../backend/data/features_9')
PROCESSED_DIR = Path('../backend/data/processed')
MFE_MODEL_PATH = Path('../backend/models_9/mfe_q50_8h/model_1H_Q50.joblib')
SAVE_DIR      = Path('../backend/models_9/dir_q50_8h')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Config ────────────────────────────────────────────────────────────────────
TRAIN_END    = '2024-06-30'
MFE_THRESH   = 30          # only train/evaluate on bars where MFE model predicts >= this
N_FOLDS      = 5
TEST_RATIO   = 0.10        # fraction of training data per CV fold

# Same VOL_DROP as MFE training notebook
VOL_DROP = [
    'atr_24', 'atr_72', 'atr_6', 'atr_ratio_6_24', 'atr_ratio_6_72',
    'vol_regime_5d', 'vol_regime_10d', 'vol_trend',
    'range_width_24', 'range_width_48', 'range_width_5d',
    'rv_zscore_24',
]

QUANTILES = [0.25, 0.50, 0.75]

print('Config ready.')
print(f'  MFE threshold : {MFE_THRESH} pips')
print(f'  Train cutoff  : {TRAIN_END}')
print(f'  CV folds      : {N_FOLDS}')
print(f'  Quantiles     : {QUANTILES}')

Config ready.
  MFE threshold : 30 pips
  Train cutoff  : 2024-06-30
  CV folds      : 5
  Quantiles     : [0.25, 0.5, 0.75]


## 1. Load Data

In [7]:
# ── Load MFE model ────────────────────────────────────────────────────────────
mfe_bundle   = joblib.load(MFE_MODEL_PATH)
mfe_model    = mfe_bundle['model']
feature_cols = mfe_bundle['feature_cols']   # 308 features — reuse exactly
print(f'MFE model loaded | {len(feature_cols)} features | {mfe_bundle["n_iters"]} iters')

# ── Load all features_9 parquets ──────────────────────────────────────────────
print('\nLoading features_9 parquets...')
dfs = [pd.read_parquet(f) for f in sorted(FEATURES_DIR.glob('*_features.parquet'))]
df  = pd.concat(dfs).sort_index()
print(f'  {len(df):,} rows | {df.shape[1]} cols | pairs: {df["pair"].nunique()}')

# ── Compute 8h log return per pair from 1H OHLCV ─────────────────────────────
# Done per-pair so close prices don't bleed across pairs
print('\nComputing ret_8h from 1H OHLCV...')
parts = []
for pair, grp in df.groupby('pair'):
    ohlcv = pd.read_parquet(PROCESSED_DIR / f'{pair}_1H.parquet')[['close']].sort_index()
    ohlcv.index.name = grp.index.name
    ohlcv['ret_8h'] = np.log(ohlcv['close'].shift(-8) / ohlcv['close'])
    grp = grp.copy()
    grp['ret_8h'] = ohlcv['ret_8h'].reindex(grp.index)
    parts.append(grp)

df = pd.concat(parts).sort_index()
print(f'  ret_8h valid: {df["ret_8h"].notna().sum():,} / {len(df):,}')

# ── Train / test split ────────────────────────────────────────────────────────
df_train = df[df.index <= TRAIN_END].copy()
df_test  = df[df.index  > TRAIN_END].copy()
print(f'\nTrain rows : {len(df_train):,}  (≤ {TRAIN_END})')
print(f'Test rows  : {len(df_test):,}   (> {TRAIN_END})')

MFE model loaded | 308 features | 1801 iters

Loading features_9 parquets...
  1,518,020 rows | 327 cols | pairs: 15

Computing ret_8h from 1H OHLCV...
  ret_8h valid: 1,471,495 / 1,518,020

Train rows : 1,378,758  (≤ 2024-06-30)
Test rows  : 139,262   (> 2024-06-30)


## 2. Filter Training Set via MFE Model (no leakage)

In [8]:
# Forward pass of the fixed MFE model on the train set.
# No leakage: MFE model was already trained on this data (it's fixed),
# and we're only using its predictions as a filter — not as a feature.
print('Running MFE model forward pass on train set...')
X_tr_all = df_train[feature_cols].ffill().fillna(0)
df_train['q50_mfe'] = mfe_model.predict(X_tr_all)

# Filter: only bars where MFE model says big move is coming AND ret_8h is valid
mask_train = (df_train['q50_mfe'] >= MFE_THRESH) & df_train['ret_8h'].notna()
df_dir_train = df_train[mask_train].copy()

print(f'Train bars total       : {len(df_train):,}')
print(f'Train bars after filter: {len(df_dir_train):,}  ({len(df_dir_train)/len(df_train):.1%} kept)')
print(f'\nTarget (ret_8h) stats on filtered train set:')
r = df_dir_train['ret_8h']
print(f'  Mean   : {r.mean():+.6f}  ({r.mean()*10000:+.2f} bps)')
print(f'  Std    : {r.std():.6f}   ({r.std()*10000:.2f} bps)')
print(f'  Skew   : {r.skew():+.4f}')
print(f'  % pos  : {(r > 0).mean():.2%}')

Running MFE model forward pass on train set...
Train bars total       : 1,378,758
Train bars after filter: 423,050  (30.7% kept)

Target (ret_8h) stats on filtered train set:
  Mean   : -0.000003  (-0.03 bps)
  Std    : 0.005163   (51.63 bps)
  Skew   : -0.3300
  % pos  : 50.51%


## 3. Walk-Forward CV — Determine Best Iterations per Quantile

In [ ]:
def pinball_loss(y_true, y_pred, alpha):
    """Mean pinball loss at quantile alpha."""
    e = y_true - y_pred
    return np.where(e >= 0, alpha * e, (alpha - 1) * e).mean()


def make_lgbm_params(alpha):
    return {
        'objective':         'quantile',
        'alpha':             alpha,
        'metric':            'quantile',
        'boosting_type':     'gbdt',
        'n_estimators':      5000,
        'learning_rate':     0.02,
        'num_leaves':        64,
        'max_depth':         6,
        'min_child_samples': 50,
        'feature_fraction':  0.7,
        'bagging_fraction':  0.8,
        'bagging_freq':      5,
        'reg_alpha':         0.1,
        'reg_lambda':        0.1,
        'random_state':      42,
        'n_jobs':            -1,
        'device':            'gpu',
        'verbose':           -1,
    }


# ── Build expanding-window folds using integer positions ─────────────────────
# reset_index so there are no duplicate labels — all slicing by integer position
df_dir_train_sorted = df_dir_train.sort_index().reset_index(drop=False)  # keep datetime as column
# We'll restore it later; for now work with RangeIndex

n              = len(df_dir_train_sorted)
fold_test_size = int(n * TEST_RATIO)

folds_pos = []
for i in range(N_FOLDS):
    train_end = n - (N_FOLDS - i) * fold_test_size
    test_end  = train_end + fold_test_size
    if train_end < fold_test_size:
        continue
    folds_pos.append((slice(0, train_end), slice(train_end, test_end)))

print(f'CV folds: {len(folds_pos)}')
idx_col = df_dir_train_sorted.columns[0]   # the datetime column after reset
for i, (tr, te) in enumerate(folds_pos):
    tr_dates = df_dir_train_sorted[idx_col].iloc[tr]
    te_dates = df_dir_train_sorted[idx_col].iloc[te]
    print(f'  Fold {i+1}: train {tr_dates.iloc[0].date()} -> {tr_dates.iloc[-1].date()} ({tr.stop:,}) '
          f'| test {te_dates.iloc[0].date()} -> {te_dates.iloc[-1].date()} ({te.stop - te.start:,})')

In [ ]:
# ── Run CV for each quantile ──────────────────────────────────────────────────
cv_results = {}

# Pure integer-indexed arrays — zero duplicate-label risk
X_all = df_dir_train_sorted[feature_cols].ffill().fillna(0).to_numpy()
y_all = df_dir_train_sorted['ret_8h'].to_numpy()

for alpha in QUANTILES:
    print(f'\n{"="*60}')
    print(f'  CV for Q{int(alpha*100)} (alpha={alpha})')
    print(f'{"="*60}')

    oof_preds  = np.full(n, np.nan)
    best_iters = []
    fold_losses = []

    for fold_i, (tr, te) in enumerate(folds_pos):
        X_tr, y_tr = X_all[tr], y_all[tr]
        X_te, y_te = X_all[te], y_all[te]

        params = make_lgbm_params(alpha)
        mdl = lgb.LGBMRegressor(**params)
        mdl.fit(
            X_tr, y_tr,
            eval_set=[(X_te, y_te)],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )

        preds = mdl.predict(X_te)
        oof_preds[te] = preds

        loss = pinball_loss(y_te, preds, alpha)
        best_iters.append(mdl.best_iteration_)
        fold_losses.append(loss)

        print(f'  Fold {fold_i+1}: iters={mdl.best_iteration_:,}  pinball={loss:.6f}  '
              f'sign_acc={((preds > 0) == (y_te > 0)).mean():.3f}')

    avg_iters = int(round(np.mean(best_iters)))
    avg_loss  = np.mean(fold_losses)

    valid     = ~np.isnan(oof_preds)
    sign_acc  = ((oof_preds[valid] > 0) == (y_all[valid] > 0)).mean()
    corr      = np.corrcoef(oof_preds[valid], y_all[valid])[0, 1]

    cv_results[alpha] = {
        'oof_preds':  oof_preds,
        'best_iters': best_iters,
        'avg_iters':  avg_iters,
        'fold_losses': fold_losses,
        'cv_pinball': avg_loss,
    }

    print(f'\n  Q{int(alpha*100)} Summary:')
    print(f'    Best iters per fold : {best_iters}')
    print(f'    Avg iters (final)   : {avg_iters}')
    print(f'    CV pinball          : {avg_loss:.6f}')
    print(f'    OOF sign accuracy   : {sign_acc:.3f}')
    print(f'    OOF corr (pred/act) : {corr:.4f}')

## 4. Train Final Models on Full Training Set

In [ ]:
print('Training final models on full training set...\n')

final_models = {}

for alpha in QUANTILES:
    avg_iters = cv_results[alpha]['avg_iters']
    params    = make_lgbm_params(alpha)
    params['n_estimators'] = avg_iters
    params.pop('device', None)

    mdl = lgb.LGBMRegressor(**params)
    mdl.fit(X_all, y_all, callbacks=[lgb.log_evaluation(period=-1)])

    final_models[alpha] = mdl

    bundle = {
        'model':        mdl,
        'quantile':     alpha,
        'feature_cols': feature_cols,
        'train_end':    TRAIN_END,
        'mfe_thresh':   MFE_THRESH,
        'n_iters':      avg_iters,
        'cv_pinball':   cv_results[alpha]['cv_pinball'],
    }
    fname = SAVE_DIR / f'model_1H_Q{int(alpha*100)}.joblib'
    joblib.dump(bundle, fname)
    size_mb = fname.stat().st_size / 1e6
    print(f'  Q{int(alpha*100)}: {avg_iters} iters | pinball={cv_results[alpha]["cv_pinball"]:.6f} | saved ({size_mb:.1f} MB)')

print(f'\nAll models saved to {SAVE_DIR}')

## 5. OOF Calibration Check

In [ ]:
print('OOF Quantile Coverage:')
print(f'  {"Quantile":<10} {"Target":>8} {"Actual":>8} {"Gap":>8}')
print(f'  {"-"*40}')

for alpha in QUANTILES:
    oof   = cv_results[alpha]['oof_preds']
    valid = ~np.isnan(oof)
    coverage = (y_all[valid] <= oof[valid]).mean()
    gap      = coverage - alpha
    flag     = '  NEEDS CALIBRATION' if abs(gap) > 0.03 else '  OK'
    print(f'  Q{int(alpha*100):<9} {alpha:>8.2f} {coverage:>8.3f} {gap:>+8.3f}{flag}')

## 6. Test Set Evaluation

In [ ]:
# ── Filter test set through MFE model ────────────────────────────────────────
print('Running MFE model on test set...')
X_te_all = df_test[feature_cols].ffill().fillna(0)
df_test['q50_mfe'] = mfe_model.predict(X_te_all)

mask_test  = (df_test['q50_mfe'] >= MFE_THRESH) & df_test['ret_8h'].notna()
df_dir_test = df_test[mask_test].copy()
print(f'Test bars after filter: {len(df_dir_test):,}  ({len(df_dir_test)/len(df_test):.1%} kept)')
print(f'Test period: {df_dir_test.index.min().date()} -> {df_dir_test.index.max().date()}')

X_te = df_dir_test[feature_cols].ffill().fillna(0)
y_te = df_dir_test['ret_8h']

# ── Predict all quantiles ─────────────────────────────────────────────────────
for alpha in QUANTILES:
    df_dir_test[f'pred_q{int(alpha*100)}'] = final_models[alpha].predict(X_te)

# ── Overall stats ─────────────────────────────────────────────────────────────
print(f'\n{"="*70}')
print(f'  TEST SET RESULTS  (post {TRAIN_END}, MFE >= {MFE_THRESH})')
print(f'{"="*70}')

for alpha in QUANTILES:
    col      = f'pred_q{int(alpha*100)}'
    preds    = df_dir_test[col].values
    actuals  = y_te.values
    pb       = pinball_loss(actuals, preds, alpha)
    coverage = (actuals <= preds).mean()
    corr     = np.corrcoef(preds, actuals)[0, 1]
    sign_acc = ((preds > 0) == (actuals > 0)).mean()
    print(f'\n  Q{int(alpha*100)}:')
    print(f'    Pinball loss  : {pb:.6f}  (CV was {cv_results[alpha]["cv_pinball"]:.6f})')
    print(f'    Coverage      : {coverage:.3f}  (target {alpha:.2f})')
    print(f'    Corr pred/act : {corr:.4f}')
    print(f'    Sign accuracy : {sign_acc:.3f}  (null=0.500)')

In [ ]:
# ── Sign accuracy by prediction confidence (Q50 deciles) ─────────────────────
print(f'\n--- Sign accuracy by |Q50 pred| decile ---')
print(f'  (does higher |prediction| -> higher sign accuracy?)')
df_dir_test['abs_q50'] = df_dir_test['pred_q50'].abs()
df_dir_test['decile']  = pd.qcut(df_dir_test['abs_q50'], q=10, labels=False)
df_dir_test['sign_correct'] = ((df_dir_test['pred_q50'] > 0) == (y_te > 0))

print(f'  {"Decile":>7} {"N":>6} {"Avg|Q50|_bps":>14} {"Sign_acc":>10}')
print(f'  {"-"*45}')
for d in range(10):
    sub = df_dir_test[df_dir_test['decile'] == d]
    if len(sub) < 10: continue
    print(f'  {d:>7} {len(sub):>6,} {sub["abs_q50"].mean()*10000:>14.2f} {sub["sign_correct"].mean():>10.3f}')

# ── Interval width as uncertainty measure ─────────────────────────────────────
# Q75 - Q25 = prediction interval width; narrower = more confident
df_dir_test['interval_width'] = df_dir_test['pred_q75'] - df_dir_test['pred_q25']
df_dir_test['width_decile']   = pd.qcut(df_dir_test['interval_width'], q=10, labels=False)

print(f'\n--- Sign accuracy by interval width decile ---')
print(f'  (narrower interval = more confident = better sign accuracy?)')
print(f'  {"Decile":>7} {"N":>6} {"Width_bps":>11} {"Sign_acc":>10}  note')
print(f'  {"-"*50}')
for d in range(10):
    sub = df_dir_test[df_dir_test['width_decile'] == d]
    if len(sub) < 10: continue
    note = '<-- narrowest' if d == 0 else ('<-- widest' if d == 9 else '')
    print(f'  {d:>7} {len(sub):>6,} {sub["interval_width"].mean()*10000:>11.2f} '
          f'{sub["sign_correct"].mean():>10.3f}  {note}')

In [ ]:
# ── Per-pair breakdown ────────────────────────────────────────────────────────
print(f'\n--- Per-pair (Q50 sign accuracy, test set) ---')
print(f'  {"Pair":<10} {"N":>6} {"% pos actual":>14} {"Q50 sign_acc":>14} {"Corr":>8}')
print(f'  {"-"*58}')
for pair, grp in df_dir_test.groupby('pair'):
    act  = y_te.loc[grp.index]
    pred = grp['pred_q50']
    if len(act) < 30: continue
    sign_acc = ((pred > 0) == (act > 0)).mean()
    corr     = np.corrcoef(pred.values, act.values)[0, 1] if len(act) > 2 else np.nan
    print(f'  {pair:<10} {len(act):>6,} {(act>0).mean():>14.3f} {sign_acc:>14.3f} {corr:>8.4f}')

# ── Sign accuracy by hour ─────────────────────────────────────────────────────
print(f'\n--- Sign accuracy by hour (Q50, test set) ---')
print(f'  {"Hour":>5} {"N":>6} {"Sign_acc":>10}')
print(f'  {"-"*25}')
df_dir_test['hour'] = df_dir_test.index.hour
for h in range(24):
    sub = df_dir_test[df_dir_test['hour'] == h]
    if len(sub) < 20: continue
    act  = y_te.loc[sub.index]
    pred = sub['pred_q50']
    print(f'  {h:>5} {len(sub):>6,} {((pred > 0) == (act > 0)).mean():>10.3f}')

## 7. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(22, 8))
fig.suptitle('Feature Importance — Directional Meta Models', fontsize=14)

for ax, alpha in zip(axes, QUANTILES):
    mdl = final_models[alpha]
    imp = pd.Series(mdl.feature_importances_, index=feature_cols).sort_values(ascending=False).head(25)
    imp[::-1].plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Q{int(alpha*100)} — Top 25 Features')
    ax.set_xlabel('Importance (split)')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved feature_importance.png')

## 8. Summary

In [ ]:
print('=' * 70)
print('  DIRECTIONAL META MODEL — TRAINING COMPLETE')
print('=' * 70)
print(f'\nModels saved to: {SAVE_DIR.resolve()}')
for alpha in QUANTILES:
    f = SAVE_DIR / f'model_1H_Q{int(alpha*100)}.joblib'
    print(f'  model_1H_Q{int(alpha*100)}.joblib  {f.stat().st_size/1e6:.1f} MB')

print(f'\n-- CV Results --')
for alpha in QUANTILES:
    r = cv_results[alpha]
    print(f'  Q{int(alpha*100)}: pinball={r["cv_pinball"]:.6f}, avg_iters={r["avg_iters"]}')

print(f'\n-- Test Set Results (UNSEEN, after {TRAIN_END}) --')
for alpha in QUANTILES:
    col      = f'pred_q{int(alpha*100)}'
    preds    = df_dir_test[col].values
    actuals  = y_te.values
    sign_acc = ((preds > 0) == (actuals > 0)).mean()
    corr     = np.corrcoef(preds, actuals)[0, 1]
    coverage = (actuals <= preds).mean()
    print(f'  Q{int(alpha*100)}: sign_acc={sign_acc:.4f}  corr={corr:.4f}  coverage={coverage:.3f} (target {alpha:.2f})')

print(f'\nFilter     : MFE Q50 >= {MFE_THRESH} pips')
print(f'Train bars : {len(df_dir_train):,}  (out of {len(df_train):,} total train)')
print(f'Test bars  : {len(df_dir_test):,}   (out of {len(df_test):,} total test)')
print(f'Features   : {len(feature_cols)}')
print(f'Test period: {df_dir_test.index.min().date()} -> {df_dir_test.index.max().date()}')